# Model Context Protocol (MCP)
MCP is an open standard for exposing tools to LLMs. An adapter connects to any MCP server and turns its tools into ordinary LangChain tools your agent can call.

## Concept
MCPAdapter connects to one or many servers and infers the transport — a local script (stdio) or a URL (streamable HTTP).
await adapter.list_tools() returns tools that drop straight into create_agent.

uv add "langchain[mcp]"

In [ ]:
import asyncio
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import dynamic_prompt, ModelRequest

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

async def main():
    config = {
        "mcpServers":{
            "weather":{"command": "python", "args": ["weather_server.py"]},  # stdio
            # "docs": {"url": "https://example.com/mcp"},                     # http
        }
    }

    async with MCPAdapter(config) as mcpAdapter:
        tools = await mcpAdapter.list_tools()
        agent = create_agent(model=model, tools=tools)
        result = await agent.ainvoke({
            "messages": [{"role": "user", "content": "What's the forecast for Oslo?"}]
        })
        print(result["messages"][-1].content)

await main()

RuntimeError: Client failed to connect: fileno

## A single stdio server
Point the adapter at a local script path — the transport is inferred as stdio.

In [ ]:
from pathlib import Path


async def main():
    async with MCPAdapter(Path("weather_server.py")) as adapter:
        agent = create_agent(model=model, tools=await adapter.list_tools())
        out = await agent.ainvoke(
            {"messages": [{"role": "user", "content": "Forecast for Oslo?"}]}
        )
        print(out["messages"][-1].content)

await main()

## A single HTTP server
A URL string is treated as a streamable-HTTP endpoint.

In [ ]:
async with MCPAdapter("https://example.com/mcp") as adapter:
    tools = await adapter.list_tools()
    print([t.name for t in tools])   # see what the server exposes
    agent = create_agent(model=model, tools = tools)

## Mix MCP tools with your own
MCP tools are ordinary LangChain tools, so combine them with local @tool functions.

In [ ]:
from langchain.tools import tool

@tool
def to_fahrenheit(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32

config = {
        "mcpServers":{
            "weather":{"command": "python", "args": ["weather_server.py"]},  # stdio
            "docs": {"url": "https://example.com/mcp"},                     # http
        }
    }

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools() + [to_fahrenheit]
    agent = create_agent("claude-sonnet-4-6", tools)

## Persist for server prompts (elicitation)
If a server pauses to ask the user something, the run needs a checkpointer to wait — same pattern as human-in-the-loop.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

async with MCPAdapter(config) as adapter:
    agent = create_agent(
        "claude-sonnet-4-6",
        await adapter.list_tools(),
        checkpointer=InMemorySaver(),
    )
    config_run = {"configurable": {"thread_id": "mcp-1"}}
    await agent.ainvoke({"messages": [{"role": "user", "content": "Book a table"}]}, config_run)

The langchain.mcp namespace is beta. If you're on an older setup, the standalone langchain-mcp-adapters package (with MultiServerMCPClient) does the same job — uv add langchain-mcp-adapters, then tools = await client.get_tools().